한국어 영화 리뷰 - BERT

In [2]:
import pandas as pd
url = "https://drive.google.com/uc?id=1KOKgZ4qCg49bgj1QNTwk1Vd29soeB27o"
df = pd.read_csv(url)

In [3]:
# rating 6 이상이면 긍정 라벨 생성 y로 저장
import numpy as np
from sklearn.model_selection import train_test_split
# y를 생성해서 review 컬럼이 x 데이터 분할 데이터 갯수 확인
y = np.array([1 if value >= 6 else 0 for value in df.rating])
X = df.review.values
# 데이터셋을 학습 검증 평가로 나눈다 x_train x_val x_test
X_,x_test, y_, y_test = train_test_split(X,y,test_size=0.2, random_state=42, stratify=y)
x_train, x_val, y_train,y_val = train_test_split(X_,y_,test_size=0.2, random_state=42, stratify=y_)
print(x_train.shape, x_val.shape, x_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

(9424,) (2356,) (2945,)
(9424,) (2356,) (2945,)


In [4]:
%pip install evaluate


Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
# from datasets import load_metric  #  2022 이후로 HuggingFace에서 deprecated   evaluate
# old system -> legacy system
import evaluate
metric = evaluate.load('accuracy')

c:\Users\sally\anaconda3\envs\p311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def compute_metrics(eval_pred):
  '''
  Args:
    eval_pred : logits,labels를 가지고있는 dataset
  Returns:
    accuracy
  '''
  logits, labels = eval_pred
  predictions =  np.argmax(logits,axis=-1)
  return metric.compute(predictions=predictions, references=labels)

In [ ]:
# 데이터셋 생성 - 클래스(상속)
# __init__ __getitem__ __len__
# x,y 각각 텐서로 만들어줍니다. y의 개수

class OurDataset(torch.utils.data.Dataset):
    def __init__(self, encodings,labels):
        self.encodings = encodings
        self.labels = labels 
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
    def __len__(self):
        return len(self.labels)

In [ ]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')   # 다국어 지원(한국어)
print(tokenizer.tokenize("안녕하세요. 반갑습니다."))

c:\Users\sally\anaconda3\envs\p311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sally\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [9]:
inputs = tokenizer("안녕하세요. 반갑습니다.")
inputs

{'input_ids': [101, 9521, 118741, 35506, 24982, 48549, 119, 9321, 118610, 119081, 48345, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [11]:
# 2022년 이후로는 Auto~~~ 토크나이져와 모델을 사용하도록 권장(벤더사에서 업데이트시 유리)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
inputs = tokenizer("안녕하세요. 반갑습니다.")
inputs = tokenizer("안녕하세요. 반갑습니다.")
print(inputs)

{'input_ids': [101, 9521, 118741, 35506, 24982, 48549, 119, 9321, 118610, 119081, 48345, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [16]:
# mBERT + Trainer 로 미세조정(Fine Tuning)
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
# 토큰화
train_encodings = tokenizer(x_train.tolist(), truncation=True, padding=True, return_tensors = 'pt')
val_input = tokenizer(x_val.tolist(), truncation=True, padding=True, return_tensors = 'pt')
test_input = tokenizer(x_test.tolist(), truncation=True, padding=True, return_tensors = 'pt')

# Dataset 생성
train_dataset = OurDataset(train_input, y_train)
val_dataset = OurDataset(val_input, y_val)
test_dataset = OurDataset(test_input, y_test)
# 분류모델 생성
model = AutoModelForSequenceClassification.from_pretrained('bert-base-multilingual-cased')
# Trainer에 사용할 Argument 셋팅
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    eval_strategy='steps'
    eval_steps=500,
    report
)
# Trainer 객체생성 및 학습(미세조정)
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)
trainer.train()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3275892784.py, line 18)

In [14]:
train_input

NameError: name 'train_input' is not defined